# Stage 2: Shared Geometry and Reference-Faithful UVDoc Inference

**Document Unwarping · 21 November 2026**

This notebook is an integration and verification driver. The actual geometry, model adapter, artifact runner, and parity verifier live in tracked Python source modules. No training, metric scoring, hybrid experiment, or dataset modification is performed.

**Run cells in order.** Use a Colab GPU runtime (preferably L4 for comparison with the earlier pilot). The source patch is installed to your existing Google Drive repository once. The original UVDoc source is cloned into `/content/`, outside your team repository. Re-running the patch cell skips already identical files. All existing changed source files receive backups before replacement. Never use `!git add` or other Git mutations here.

## 01. Mount the existing project

In [1]:
# Mount the existing Google Drive repository.
from pathlib import Path
import os, sys, subprocess, json
from datetime import datetime, timezone
from uuid import uuid4

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Document Unwarping")
assert ROOT.is_dir() and (ROOT / "data/manifests.py").is_file(), "Expected the Stage 1 repository in Google Drive"
os.environ["DOCUMENT_UNWARPING_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Stage 1 source available:", (ROOT / "models/interfaces.py").is_file())

Mounted at /content/drive
Project root: /content/drive/MyDrive/Document Unwarping
Stage 1 source available: True


## 02. Apply the Stage 2 source-only patch

Download `document_unwarping_stage2_patch.zip` from the ChatGPT handoff, then select it in the upload dialog. This cell verifies every file checksum and path before writing. It never touches `dataset/`, existing notebooks, checkpoints, the team Git history, or prior run artifacts.

In [2]:
# Safe, backed-up installation of the supplied Stage 2 source patch.
from google.colab import files
from io import BytesIO
from zipfile import ZipFile
from pathlib import PurePosixPath
from hashlib import sha256
import shutil

PATCH_NAME = "document_unwarping_stage2_patch.zip"
uploaded = files.upload()
assert PATCH_NAME in uploaded, f"Please select {PATCH_NAME}"
archive_bytes = BytesIO(uploaded[PATCH_NAME])

with ZipFile(archive_bytes) as archive:
    member_names = archive.namelist()
    assert "stage2_manifest.json" in member_names
    manifest = json.loads(archive.read("stage2_manifest.json"))
    declared = manifest["files"]
    assert set(member_names) == set(declared) | {"stage2_manifest.json"}, "Patch file inventory mismatch"
    for name, expected_hash in declared.items():
        parts = PurePosixPath(name).parts
        assert parts and ".." not in parts and not name.startswith("/") and "\\" not in name and ":" not in name
        allowed = (name.startswith("geometry/") or name.startswith("models/deep/") or
                   name.startswith("configs/models/") or name.startswith("tests/test_geometry_") or
                   name in {"scripts/run_inference.py", "scripts/verify_uvdoc_reference.py",
                            "tests/test_uvdoc_adapter.py", "tests/test_uvdoc_reference.py",
                            "tests/test_inference_runner.py", "documentation/geometry_conventions.md",
                            "documentation/stage2_implementation.md", "requirements-stage2.txt",
                            "notebooks/[21.11] Stage_2_Geometry_and_UVDoc_Inference.ipynb"})
        assert allowed, f"Unexpected path in patch: {name}"
        assert sha256(archive.read(name)).hexdigest() == expected_hash, f"Checksum mismatch: {name}"

    BACKUP_DIR = ROOT / "runs/stage2_patch_backups" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    for name in sorted(declared):
        target = ROOT.joinpath(*PurePosixPath(name).parts)
        content = archive.read(name)
        if target.is_file() and target.read_bytes() == content:
            print("Already current:", name)
            continue
        if target.exists():
            backup = BACKUP_DIR.joinpath(*PurePosixPath(name).parts)
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
        print("Installed:", name)
print("Patch installation complete. Backups if needed:", BACKUP_DIR)

KeyboardInterrupt: 

In [3]:
# Stage 2 files were already extracted into the repository.
required_files = [
    "geometry/coordinates.py",
    "geometry/grids.py",
    "geometry/homography.py",
    "geometry/sampling.py",
    "geometry/composition.py",
    "models/deep/uvdoc.py",
    "scripts/run_inference.py",
    "scripts/verify_uvdoc_reference.py",
    "configs/models/uvdoc_official.yaml",
    "requirements-stage2.txt",
]

missing = [p for p in required_files if not (ROOT / p).is_file()]

assert not missing, f"Missing Stage 2 files: {missing}"

print("Stage 2 source files found. Patch installation skipped.")

Stage 2 source files found. Patch installation skipped.


## 03. Environment check and optional non-Torch dependencies

Keep the CUDA-enabled PyTorch build already provided by Colab. Only install missing non-Torch packages. The optional `hdf5storage` package is needed if benchmark `.mat` grid export is enabled.

In [4]:
# Do not blindly replace Colab's CUDA/PyTorch environment.
import importlib.util
required = {"cv2": "opencv-python-headless>=4.8,<5", "yaml": "PyYAML>=6,<7",
            "PIL": "Pillow>=10,<13", "h5py": "h5py>=3.9,<4",
            "pytest": "pytest>=8,<10", "scipy": "scipy>=1.11,<2",
            "hdf5storage": "hdf5storage>=0.2,<0.3"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing only missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch, cv2, numpy as np, yaml, h5py
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("OpenCV:", cv2.__version__, "NumPy:", np.__version__, "h5py:", h5py.__version__)
assert torch.cuda.is_available(), "Switch to a CUDA GPU runtime for reference parity and inference"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

Installing only missing packages: ['hdf5storage>=0.2,<0.3']
Python: 3.13.16
PyTorch: 2.14.1+cu130 CUDA runtime: 13.0
OpenCV: 4.14.0 NumPy: 2.4.6 h5py: 3.14.0
GPU: NVIDIA L4


## 04. Run Stage 1 and Stage 2 automated tests

All geometry tests run on CPU; they must pass even before we obtain official weights.

In [5]:
# Verify the entire repo's current tests, including original Stage 1 tests.
test_result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
                             cwd=str(ROOT), text=True, capture_output=True)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0, "The test suite did not pass"
print("All automated tests passed in Colab.")

....................................................................     [100%]
68 passed in 40.19s

All automated tests passed in Colab.


## 05. Verify Stage 1 real-data access

Some UVDoc Final assets are archived. This cell stages **only the first three selected records** if files are missing. Nothing is written into the original dataset directory. The next cell repeats the physical smoke test.

In [6]:
# Selectively stage only necessary UVDoc Final members from the existing archive.
from zipfile import ZipFile
from pathlib import PurePosixPath
from data.manifests import load_records
from data.paths import PathResolver

UVFINAL_ARCHIVE = ROOT / "dataset/UVDoc/_archives/UVDoc_final.zip"
RUNTIME_STAGE = Path("/content/uvdoc_runtime")
resolver = PathResolver(ROOT)
selected_final = load_records("uvdoc_final", ROOT)[:3]
missing_paths = []
for record in selected_final:
    for value in [record.input_path, *record.annotation_paths.values()]:
        if resolver.find(value) is None:
            missing_paths.append(value)
missing_paths = sorted(set(missing_paths))
print("UVDoc Final files to stage:", len(missing_paths))
if missing_paths:
    assert UVFINAL_ARCHIVE.is_file(), f"Missing original archive: {UVFINAL_ARCHIVE}"
    with ZipFile(UVFINAL_ARCHIVE) as archive:
        for logical in missing_paths:
            assert logical.startswith("dataset/UVDoc/UVDoc_final/")
            member = logical.removeprefix("dataset/UVDoc/")
            parts = PurePosixPath(member).parts
            assert parts[0] == "UVDoc_final" and ".." not in parts and "\\" not in member
            info = archive.getinfo(member)
            assert not info.is_dir()
            destination = RUNTIME_STAGE.joinpath(*parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            partial = destination.with_suffix(destination.suffix + ".partial")
            try:
                with archive.open(info) as source, partial.open("wb") as target:
                    shutil.copyfileobj(source, target, length=8 * 1024 * 1024)
                assert partial.stat().st_size == info.file_size
                partial.replace(destination)
            finally:
                if partial.exists():
                    partial.unlink()
            print("Staged:", member)
resolver = PathResolver(ROOT)
assert all(resolver.find(path) is not None for path in missing_paths)
print("Representative UVDoc Final inputs now available.")

UVDoc Final files to stage: 8
Staged: UVDoc_final/grid2d/00_00000_1_0_0.mat
Staged: UVDoc_final/grid2d/00_00000_1_0_1.mat
Staged: UVDoc_final/grid2d/00_00000_1_1_0.mat
Staged: UVDoc_final/img/00000.png
Staged: UVDoc_final/img/00001.png
Staged: UVDoc_final/img/00002.png
Staged: UVDoc_final/metadata_sample/00000.json
Staged: UVDoc_final/metadata_sample/00001.json
Representative UVDoc Final inputs now available.


In [7]:
# Final Stage 1 real-data smoke gate.
GATE_RUN = ROOT / "runs/data_validation" / ("stage2_stage1_gate_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:6])
check = subprocess.run([sys.executable, "scripts/validate_datasets.py", "--root", str(ROOT),
                        "--check-files", "--max-files-per-dataset", "3", "--smoke-read", "2",
                        "--output", str(GATE_RUN)], cwd=str(ROOT), text=True, capture_output=True)
print(check.stdout)
if check.stderr:
    print(check.stderr)
print((GATE_RUN / "report.md").read_text(encoding="utf-8"))
assert check.returncode == 0, "Stage 1 real-data gate incomplete; inspect report before inference"
print("Stage 1 real-data smoke gate passed.")

Dataset validation report: /content/drive/MyDrive/Document Unwarping/runs/data_validation/stage2_stage1_gate_20261121T104245Z_2d67d8/report.md
uvdoc_final: 20000 rows, PASS, 0 unavailable physical files
uvdoc_benchmark: 50 rows, PASS, 0 unavailable physical files
docunet: 130 rows, PASS, 0 unavailable physical files
dir300: 300 rows, PASS, 0 unavailable physical files
saved UVDoc split: PASS
smoke read: {'PASS': 8, 'UNAVAILABLE': 0, 'FAIL': 0, 'ENVIRONMENT_ERROR': 0}

# Stage 1 dataset validation

This report checks the existing EDA manifests without modifying source data.
No neural inference, benchmark scores, or geometric performance results are implied.

Project root: `/content/drive/MyDrive/Document Unwarping`

## Inventory

| Dataset | Records | Groups | Integrity | Files checked | Files unavailable |
|---|---:|---:|---|---:|---:|
| UVDoc Final | 20,000 | 1,008 | PASS | 12 | 0 |
| UVDoc Benchmark | 50 | 50 | PASS | 12 | 0 |
| DocUNet | 130 | 65 | PASS | 6 | 0 |
| DIR300 | 300 | 30

## 06. Obtain and verify the exact official UVDoc source

The pinned authors' checkout stays in the temporary Colab filesystem. We do not clone inside `Document Unwarping/`. If this external directory already exists with another revision, the cell raises rather than silently changing its contents.

In [8]:
# Clone once into /content, check out the pinned commit, and verify the checkpoint.
from models.deep.uvdoc import load_uvdoc_config, verify_source, hash_file

OFFICIAL_CFG = load_uvdoc_config()
COMMIT = OFFICIAL_CFG["source"]["commit"]
OFFICIAL_SOURCE = Path("/content/uvdoc_reference_stage2")
if not OFFICIAL_SOURCE.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", OFFICIAL_CFG["source"]["repository"],
                    str(OFFICIAL_SOURCE)], check=True)
    current_rev = subprocess.check_output(["git", "-C", str(OFFICIAL_SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if current_rev != COMMIT:
        subprocess.run(["git", "-C", str(OFFICIAL_SOURCE), "fetch", "--quiet", "--depth", "1", "origin", COMMIT], check=True)
        subprocess.run(["git", "-C", str(OFFICIAL_SOURCE), "checkout", "--quiet", "--detach", COMMIT], check=True)
source_hashes = verify_source(OFFICIAL_SOURCE, COMMIT)
CHECKPOINT = OFFICIAL_SOURCE / OFFICIAL_CFG["checkpoint"]["relative_path"]
assert CHECKPOINT.is_file(), f"Checkpoint missing: {CHECKPOINT} (inspect Git LFS download if relevant)"
assert CHECKPOINT.stat().st_size > 1024 * 1024, "Checkpoint may be an unmaterialized Git LFS pointer"
actual_sha = hash_file(CHECKPOINT)
assert actual_sha == OFFICIAL_CFG["checkpoint"]["sha256"], f"Checkpoint SHA mismatch: {actual_sha}"
print("Official commit:", COMMIT)
print("Checkpoint bytes:", CHECKPOINT.stat().st_size)
print("Checkpoint SHA256:", actual_sha)
print("Pinned inference source hashes:", source_hashes)

Official commit: 4c9b82b537057aff2526e6dd118a847cdd072e82
Checkpoint bytes: 32158393
Checkpoint SHA256: 7e90861b8a516eb4bc51f84bd889cb77275743d2d1d3ca8091951ec9f2b7da23
Pinned inference source hashes: {'model.py': '320f460edc54cf830dfdd9788dfbc1b96a836429b64ed6078a30582be6f59e42', 'utils.py': '64909e09e7493dcef1cd356b833c316e87c8ce31a544c567dee4b95ece78d3c2', 'demo.py': 'b173c64775e2d9e5f41f3cfe89adbc38e45fc62f26d3cba625e01c3fa39d3926'}


## 07. Verify architecture and numerical outputs

In [9]:
# Load weights safely and verify the expected 2D/3D model output dimensions.
from models.deep.uvdoc import load_uvdoc_model

LOADED = load_uvdoc_model(OFFICIAL_CFG, device=DEVICE, source_dir=OFFICIAL_SOURCE)
parameters = sum(p.numel() for p in LOADED.model.parameters())
assert parameters == 8_003_463
with torch.inference_mode():
    zero = torch.zeros((1, 3, 712, 488), device=DEVICE)
    grid2, grid3 = LOADED.model(zero)
assert tuple(grid2.shape) == (1, 2, 45, 31)
assert tuple(grid3.shape) == (1, 3, 45, 31)
assert bool(torch.isfinite(grid2).all() and torch.isfinite(grid3).all())
print("Model parameter count:", parameters)
print("Predicted grid shapes:", tuple(grid2.shape), tuple(grid3.shape))
del zero, grid2, grid3
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Model parameter count: 8003463
Predicted grid shapes: (1, 2, 45, 31) (1, 3, 45, 31)


## 08. First real-image inference: UVDoc Benchmark `00010`

This is an implementation diagnostic, not a published benchmark metric.

In [10]:
# Run a single image with the reusable adapter.
from models.deep.uvdoc import predict_uvdoc

benchmark_records = load_records("uvdoc_benchmark", ROOT)
SAMPLE = next(r for r in benchmark_records if r.sample_id == "00010")
SOURCE_IMAGE = PathResolver(ROOT).resolve(SAMPLE.input_path)
PREDICTION = predict_uvdoc(LOADED, SOURCE_IMAGE)
assert PREDICTION.result.status == "OK"
assert PREDICTION.result.image.dtype == np.uint8
print("Sample:", SAMPLE.sample_key)
print("Input decoded size (W,H):", PREDICTION.input_size)
print("Output size (W,H):", PREDICTION.output_size)
print("Coarse 2D/3D grids:", PREDICTION.grid2d.shape, PREDICTION.grid3d.shape)
print("Phase timings (ms):", json.dumps(PREDICTION.timings_ms, indent=2))

Sample: UVDoc_Benchmark_rendered_00010
Input decoded size (W,H): (869, 1180)
Output size (W,H): (869, 1180)
Coarse 2D/3D grids: (2, 45, 31) (3, 45, 31)
Phase timings (ms): {
  "preprocess_ms": 10.391302000016367,
  "forward_ms": 9.411946000000171,
  "warp_ms": 34.48633899984088,
  "conversion_ms": 13.936223999962749,
  "decode_ms": 2502.0606880000287
}


## 09. Independently compare against the authors' original demo

The verifier invokes the original `demo.py` **in a separate process** on a byte-for-byte temporary input copy. It separately checks the original preprocessing tensor, predicted 2D/3D grids, and the final uint8 PNG. The output gate requires pixel-exact equality.

In [11]:
# This is a strict acceptance gate, not an approximate visual comparison.
from scripts.verify_uvdoc_reference import compare_with_official_demo

PARITY_DIR = ROOT / "runs/inference" / ("stage2_parity_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:6])
PARITY = compare_with_official_demo(LOADED, SOURCE_IMAGE, PARITY_DIR)
print(json.dumps(PARITY, indent=2))
print("Parity artifacts:", PARITY_DIR)
assert PARITY["status"] == "PASS", "Stage 2 official-demo parity failed; inspect its JSON report"

{
  "source_image": "/content/drive/MyDrive/Document Unwarping/dataset/UVDoc/UVDoc_benchmark/img/00010.png",
  "source_image_sha256": "cd659ac86d3dad712912608bb636322db9f32503c9b7c715fc3448f559476eda",
  "source_commit": "4c9b82b537057aff2526e6dd118a847cdd072e82",
  "checkpoint_sha256": "7e90861b8a516eb4bc51f84bd889cb77275743d2d1d3ca8091951ec9f2b7da23",
  "device": "cuda",
  "preprocess_max_abs": 0.0,
  "grid2d_max_abs": 0.0,
  "grid3d_max_abs": 0.0,
  "output_max_abs_uint8": 0,
  "output_mean_abs_uint8": 0.0,
  "status": "PASS",
  "reason": "All reference equality gates passed"
}
Parity artifacts: /content/drive/MyDrive/Document Unwarping/runs/inference/stage2_parity_20261121T104325Z_87f5ac


## 10. Execute a fixed multi-dataset diagnostic subset

These IDs have already appeared in the pilot or Stage 1 inspection. Selection is explicit, and no real-world reference images are used for inference or tuning. Each dataset/variant receives its own immutable run folder.

In [12]:
# Use one loaded model for all selected diagnostic datasets.
from scripts.run_inference import run_records, selected_records
from data.manifests import manifest_path

RUN_LABEL = "stage2_integration_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:6]
RUN_PARENT = ROOT / "runs/inference" / RUN_LABEL
RUN_PARENT.mkdir(parents=True, exist_ok=False)
SAVE_DENSE_MAPS = False          # Full-resolution dense maps are optional and large.
EXPORT_BENCHMARK_MAT = False     # Set True if you explicitly need official bm files.
DIAGNOSTIC_CASES = [
    ("uvdoc_benchmark", None, ["00010", "00011"]),
    ("uvdoc_final", None, ["00000", "00001"]),
    ("docunet", "crop", ["38_1", "64_1"]),
    ("docunet", "original", ["38_1"]),
    ("dir300", None, ["8", "38"]),
]
RUN_SUMMARIES = []
for dataset, variant, ids in DIAGNOSTIC_CASES:
    name = dataset + ("_" + variant if variant else "")
    records = selected_records(dataset, ROOT, sample_ids=ids, variant=variant)
    summary = run_records(records, PathResolver(ROOT), LOADED, RUN_PARENT / name,
                          hash_file(manifest_path(dataset, ROOT)),
                          save_dense_maps=SAVE_DENSE_MAPS,
                          export_benchmark_mat=EXPORT_BENCHMARK_MAT)
    RUN_SUMMARIES.append({"dataset_variant": name, **summary})
    print(name, "=>", summary)
print("Integration run folder:", RUN_PARENT)

[1/2] UVDoc_Benchmark_rendered_00010: OK
[2/2] UVDoc_Benchmark_rendered_00011: OK
uvdoc_benchmark => {'attempted': 2, 'success': 2, 'unavailable': 0, 'failed': 0, 'run_dir': '/content/drive/MyDrive/Document Unwarping/runs/inference/stage2_integration_20261121T104333Z_cca74c/uvdoc_benchmark'}
[1/2] UVDoc_Final_rendered_00000: OK
[2/2] UVDoc_Final_rendered_00001: OK
uvdoc_final => {'attempted': 2, 'success': 2, 'unavailable': 0, 'failed': 0, 'run_dir': '/content/drive/MyDrive/Document Unwarping/runs/inference/stage2_integration_20261121T104333Z_cca74c/uvdoc_final'}
[1/2] DocUNet_crop_38_1: OK
[2/2] DocUNet_crop_64_1: OK
docunet_crop => {'attempted': 2, 'success': 2, 'unavailable': 0, 'failed': 0, 'run_dir': '/content/drive/MyDrive/Document Unwarping/runs/inference/stage2_integration_20261121T104333Z_cca74c/docunet_crop'}
[1/1] DocUNet_original_38_1: OK
docunet_original => {'attempted': 1, 'success': 1, 'unavailable': 0, 'failed': 0, 'run_dir': '/content/drive/MyDrive/Document Unwarping/r

## 11. Verify saved artifacts and generate Stage 2 integration report

Saved PNGs and coarse grids are reopened. The report explicitly distinguishes source tests, actual checkpoint parity, inference coverage, and methods/evaluation that are still out of scope.

In [13]:
# Final consistency and evidence table.
import csv

for row in RUN_SUMMARIES:
    subrun = Path(row["run_dir"])
    with (subrun / "samples.csv").open(newline="", encoding="utf-8") as handle:
        attempts = list(csv.DictReader(handle))
    assert len(attempts) == row["attempted"]
    for sample in attempts:
        if sample["status"] == "OK":
            image_path = subrun / sample["image_artifact"]
            grids_path = subrun / sample["grid_artifact"]
            assert image_path.is_file() and grids_path.is_file()
            assert cv2.imread(str(image_path), cv2.IMREAD_COLOR) is not None
            with np.load(grids_path, allow_pickle=False) as grids:
                assert grids["grid2d"].shape == (2, 45, 31)
                assert grids["grid3d"].shape == (3, 45, 31)

summary_lines = ["# Stage 2 Colab integration evidence", "",
                 "This is implementation verification, not published benchmark evaluation.", "",
                 f"- Automated tests: **PASS** (notebook Cell 04)",
                 f"- Stage 1 real-data smoke gate: **PASS** (`{GATE_RUN}`)",
                 f"- Pinned official source and checkpoint: **PASS**",
                 f"- Independent official-demo parity: **{PARITY['status']}** (`{PARITY_DIR}`)",
                 "", "## Multi-dataset diagnostic inference", "",
                 "| Dataset/variant | Attempted | OK | Unavailable | Failed |",
                 "|---|---:|---:|---:|---:|"]
for row in RUN_SUMMARIES:
    summary_lines.append(f"| {row['dataset_variant']} | {row['attempted']} | {row['success']} | {row['unavailable']} | {row['failed']} |")
summary_lines.extend(["", "## Boundaries", "",
                      "No benchmark MS-SSIM, LD, AD, OCR, line score, training, hybrid, or final comparison has been run.",
                      "See each subrun's samples.csv, diagnostics, config.json, environment.json, and provenance.json.", ""])
(RUN_PARENT / "report.md").write_text("\n".join(summary_lines), encoding="utf-8")
print((RUN_PARENT / "report.md").read_text(encoding="utf-8"))
assert all(row["success"] == row["attempted"] for row in RUN_SUMMARIES), "Some physical samples did not complete. Review subrun failure tables."
print("Stage 2 full Colab integration gates passed.")

# Stage 2 Colab integration evidence

This is implementation verification, not published benchmark evaluation.

- Automated tests: **PASS** (notebook Cell 04)
- Stage 1 real-data smoke gate: **PASS** (`/content/drive/MyDrive/Document Unwarping/runs/data_validation/stage2_stage1_gate_20261121T104245Z_2d67d8`)
- Pinned official source and checkpoint: **PASS**
- Independent official-demo parity: **PASS** (`/content/drive/MyDrive/Document Unwarping/runs/inference/stage2_parity_20261121T104325Z_87f5ac`)

## Multi-dataset diagnostic inference

| Dataset/variant | Attempted | OK | Unavailable | Failed |
|---|---:|---:|---:|---:|
| uvdoc_benchmark | 2 | 2 | 0 | 0 |
| uvdoc_final | 2 | 2 | 0 | 0 |
| docunet_crop | 2 | 2 | 0 | 0 |
| docunet_original | 1 | 1 | 0 | 0 |
| dir300 | 2 | 2 | 0 | 0 |

## Boundaries

No benchmark MS-SSIM, LD, AD, OCR, line score, training, hybrid, or final comparison has been run.
See each subrun's samples.csv, diagnostics, config.json, environment.json, and provenance.

## Final handoff

Keep the generated `RUN_PARENT/report.md`, `PARITY_DIR/reference_equivalence.csv`, and `RUN_PARENT/*/samples.csv` as evidence. Source code resides in `geometry/`, `models/deep/`, and `scripts/`; the notebook should remain a lightweight driver. Share the report and parity CSV for an independent review before claiming Stage 2 is verified. No PowerShell execution is necessary.